In [1]:
"""ACCESS-C demo/test."""

%load_ext autoreload
%autoreload 2
import shutil
import xarray as xr

import thuner.data as data
import thuner.option as option
import thuner.analyze as analyze
import thuner.parallel as parallel
import thuner.visualize as visualize
import thuner.default as default
import thuner.config as config
import thuner.utils as utils


Welcome to the Thunderstorm Event Reconnaissance (THUNER) package 
v0.0.16! This package is still in testing and development. Please 
visit github.com/THUNER-project/THUNER for examples, and to report 
issues or contribute.
 
THUNER is a flexible toolkit for performing multi-feature detection, 
tracking, tagging and analysis of events within meteorological datasets. 
The intended application is to convective weather events. For examples 
and instructions, see https://github.com/THUNER-project/THUNER and 
https://thuner.readthedocs.io/en/latest/. If you use THUNER in your 
research, consider citing the following papers;

Short et al. (2023), doi: 10.1175/MWR-D-22-0146.1
Raut et al. (2021), doi: 10.1175/JAMC-D-20-0119.1
Fridlind et al. (2019), doi: 10.5194/amt-12-2979-2019
Whitehall et al. (2015), doi: 10.1007/s12145-014-0181-3
Dixon and Wiener (1993), doi: 10.1175/1520-0426(1993)010<0785:TTITAA>2.0.CO;2
Leese et al. (1971), doi: 10.1175/1520-0450(1971)010<0118:AATFOC>2.0.CO;2



In [2]:
# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True

# Parent directory for saving outputs
base_local = config.get_outputs_directory()
output_parent = base_local / f"runs/access_c/access_c_demo"
options_directory = output_parent / "options"
visualize_directory = output_parent / "visualize"

In [3]:
# Delete the output directory for the run if it already exists
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)

In [4]:
# Download the demo data
remote_directory = "s3://thuner-storage/THUNER_output/input_data/raw/ops_aps3/"
data.get_demo_data(base_local, remote_directory)

2026-06-03 20:10:04,049 - thuner.data._utils - INFO - Syncing directory /home/ewan/THUNER_output/input_data/raw/ops_aps3. Please wait.


In [5]:
# Create the dataset options
# For model datasets we generally need to specify which model run we want, in
# addition to the start and end times. Typically we want to discard spin up times.
run_start = "2021-12-01T12:00:00"  # The start time of the run we want
start = "2021-12-02T06:00:00"  # The start time of the data we want to analyze.
end = "2021-12-02T12:00:00"  # The end time of the data we want to analyze.
times_dict = {"start": start, "end": end, "run_start": run_start}

access_1km_options = data.access.AccessCOptions(
    **times_dict, name="access_1km", filename="radar_refl_1km.nc"
)
# access_maxcol shares the same native ACCESS-C grid as access_1km, so it reuses the
# regridder weights built for access_1km rather than building (and storing) its own.
access_max_col_options = data.access.AccessCOptions(
    **times_dict,
    name="access_maxcol",
    filename="maxcol_refl.nc",
    regridder_from="access_1km",
)

2026-06-03 20:10:10,592 - thuner.data.access - INFO - Generating ACCESS-C filepaths.
2026-06-03 20:10:10,593 - thuner.data.access - INFO - Generating ACCESS-C filepaths.


In [6]:
datasets=[access_1km_options, access_max_col_options]
data_options = option.data.DataOptions(datasets=datasets)
data_options.to_json(options_directory / "data.json")

grid_options = option.grid.GridOptions()
grid_options.to_json(options_directory / "grid.json")

track_options = default.track.access_c_track()
track_options.to_json(options_directory / "track.json")

2026-06-03 20:10:11,276 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.
2026-06-03 20:10:11,276 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


In [7]:
times = utils.generate_dataset_times(data_options.dataset_by_name("access_1km"))
parallel.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    output_directory=output_parent,
    dataset_name='access_1km',
    num_processes=4,
)

2026-06-03 20:10:12,179 - thuner.parallel - INFO - Beginning parallel tracking with 4 processes.
2026-06-03 20:10:12,190 - thuner.parallel - INFO - Pre-computing regridder weights for access_1km.
2026-06-03 20:10:12,191 - thuner.data.access - INFO - Converting access_1km dataset for time 2021-12-01T12:00:00.
2026-06-03 20:10:12,202 - thuner.grid - INFO - Creating new geographic grid with spacing 0.025, 0.025.
2026-06-03 20:10:12,203 - thuner.data._utils - INFO - Building regridder; this can take a while for large grids.
2026-06-03 20:10:21,536 - thuner.utils - INFO - Grid options not set. Inferring from dataset.
2026-06-03 20:10:21,950 - thuner.parallel - INFO - Verifying existing regridder weights for access_maxcol.
2026-06-03 20:10:21,951 - thuner.data.access - INFO - Converting access_maxcol dataset for time 2021-12-01T12:00:00.
2026-06-03 20:10:21,957 - thuner.grid - INFO - Creating new geographic grid with spacing 0.025, 0.025.
2026-06-03 20:10:21,958 - thuner.data._utils - INFO -

In [8]:
analysis_options = analyze.mcs.AnalysisOptions()
analysis_options.to_json(options_directory / "analysis.json")
analyze.mcs.process_velocities(output_parent, profile_dataset=None)
analyze.mcs.quality_control(output_parent, analysis_options)

2026-06-03 20:11:12,478 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


2026-06-03 20:11:12,546 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


In [9]:
style = "presentation"
attribute_handlers = default.visualize.grouped_attribute_handlers(output_parent, style)
figure_options = option.visualize.GroupedHorizontalAttributeOptions(
    name="mcs_attributes",
    object_name="mcs",
    style=style,
    attribute_handlers=attribute_handlers,
    altitude_titles=False,
)
visualize.attribute.series(
    output_directory=output_parent,
    start_time=start,
    end_time=end,
    figure_options=figure_options,
    dataset_name="access_1km",
    parallel_figure=True,
    by_date=False,
    num_processes=4,
)

2026-06-03 20:11:13,207 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.
2026-06-03 20:11:13,214 - thuner.visualize.attribute - INFO - Visualizing attributes at time 2021-12-02T06:00:00.000000000.
2026-06-03 20:11:13,276 - thuner.data.access - INFO - Converting access_1km dataset for time 2021-12-02T06:00:00.
2026-06-03 20:11:13,283 - thuner.grid - INFO - Creating new geographic grid with spacing 0.025, 0.025.
2026-06-03 20:11:13,284 - thuner.data._utils - INFO - Loading regridder weights from file.
2026-06-03 20:11:14,651 - thuner.utils - INFO - Grid options not set. Inferring from dataset.
2026-06-03 20:11:15,055 - thuner.data.access - INFO - Converting access_maxcol dataset for time 2021-12-02T06:00:00.
2026-06-03 20:11:15,061 - thuner.data._utils - INFO - Loading regridder weights from file.
2026-06-03 20:11:18,549 - thuner.visualize.attribute - INFO - Saving mcs_attributes figure for 2021-12-02T06:00:00.000000000.
2026-06-03 20:11:22,610 - th

![MCS detection and matching for ACCESS-C data.](https://raw.githubusercontent.com/THUNER-project/THUNER/refs/heads/main/gallery/mcs_access_c.gif)